In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
import asyncio
import sys

if sys.platform == "win32":
  # ✅ get_event_loop_policy() ke sath round brackets lazmi hain
  if not isinstance(
      asyncio.get_event_loop_policy(), asyncio.WindowsProactorEventLoopPolicy
  ):
    asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

  if "ipykernel" in sys.modules:
    sys.stderr = sys.__stderr__

In [5]:
import os
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "local_server": {
            "transport": "stdio",
            "command": "python",
            "args": [os.path.abspath("2.1-Server.py")],
        }
    }
)

In [6]:
tools = await client.get_tools()

resources = await client.get_resources("local_server")

In [8]:
from langchain.agents import create_agent

agent = create_agent(
    model="gpt-5-nano",
    tools=tools,
)

In [9]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = await agent.ainvoke(
    {"messages": [HumanMessage(content="Tell me about the langchain-mcp-adapters library")]},
    config=config
)

In [11]:
from pprint import pprint

pprint(response["messages"][-1].content)

('Here’s a concise overview of the LangChain MCP Adapters library and what '
 'it’s for.\n'
 '\n'
 'What it is\n'
 '- A bridge between Anthropic’s Model Context Protocol (MCP) and '
 'LangChain/LangGraph.\n'
 '- It lets you take existing MCP tool servers and use them inside LangChain '
 'agents and workflows without writing a bunch of glue code.\n'
 '- Available in two ecosystems:\n'
 '  - Python: langchain_mcp_adapters\n'
 '  - JavaScript/TypeScript: @langchain/mcp-adapters (LangChain.js)\n'
 '\n'
 'What problem it solves\n'
 '- MCP servers expose many tools via a standard protocol. Without adapters, '
 'you’d have to write custom integration for each MCP server.\n'
 '- The adapters provide a single way to:\n'
 '  - discover MCP tools across multiple servers,\n'
 '  - convert MCP tools, prompts, and resources into LangChain-friendly '
 'formats,\n'
 '  - run tools from those MCP servers inside LangChain agents, including '
 'LangGraph workflows.\n'
 '\n'
 'Key features\n'
 '- Multi-se

In [13]:
client = MultiServerMCPClient(
    {
        "time": {
            "transport": "stdio",
            "command": "python",
            "args": [
                "-m",
                "mcp_server_time",
                "--local-timezone=America/New_York",
            ],
        }
    }
)

tools = await client.get_tools()
print(tools)

[StructuredTool(name='get_current_time', description='Get current time in a specific timezone', args_schema={'type': 'object', 'properties': {'timezone': {'type': 'string', 'description': "IANA timezone name (e.g., 'America/New_York', 'Europe/London'). Use 'America/New_York' as local timezone if no timezone provided by the user."}}, 'required': ['timezone']}, metadata={'title': None, 'readOnlyHint': True, 'destructiveHint': False, 'idempotentHint': True, 'openWorldHint': False}, handle_tool_error=<function _handle_mcp_tool_error at 0x0000012A61169120>, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x0000012A61DCE660>), StructuredTool(name='convert_time', description='Convert time between timezones', args_schema={'type': 'object', 'properties': {'source_timezone': {'type': 'string', 'description': "Source IANA timezone name (e.g., 'America/New_York', 'Europe/London'). Use 'America/New_York' as local timezone if no so

In [14]:
agent = create_agent(
    model="gpt-5-nano",
    tools=tools,
)

In [15]:
question = HumanMessage(content="What time is it?")

response = await agent.ainvoke(
    {"messages": [question]}
)

pprint(response)

{'messages': [HumanMessage(content='What time is it?', additional_kwargs={}, response_metadata={}, id='bdd26799-8d35-4cad-b45f-84c2a3a344ae'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 219, 'prompt_tokens': 295, 'total_tokens': 514, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 192, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EVQDn4DZhFlntanpia94EpAKNF5Cn', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a10950-e1dc-7af1-a3e9-dff5c67480cf-0', tool_calls=[{'name': 'get_current_time', 'args': {'timezone': 'America/New_York'}, 'id': 'call_SDc9gXO9SNSc